# Coastline extraction
Extracting coastline data for the Barwon river mouth from Landsat data

In [3]:
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import arcgis
import anywidget
import arcpy
from arcpy.management import PolygonToLine
from arcgis import GIS
from arcgis.features import FeatureSet, GeoAccessor, GeoSeriesAccessor
from arcgis.geometry import Point, buffer
from arcgis.raster import ImageryLayer

BARWON_HEADS = (144.49, -38.27)
WGS84 = 4326
WEBMERCATOR = 3857
WEBMERCATOR_LATEST = 102100
M_ESRI = 9001
BUFFER_M = 10000


GIS_PROFILE = ('https://sabastianbouma.maps.arcgis.com', 'sabastianbouma', '18OceanGroveE2ab6214')
LANDSAT_URL = "https://landsat2.arcgis.com/arcgis/rest/services/Landsat/MS/ImageServer"

START, END = datetime.datetime(2016, 1, 1), datetime.datetime(2019, 12, 31)
MAX_CLOUD = 0.03
MAX_SCENES = 1

OUT_SR = 7855
CELL = 30
BAND_IDS = {"blue": 1, "red": 3, "nir": 4}

In [5]:
try:
    REPO = Path(__file__).resolve().parents[1]
except NameError:                     # notebook / interactive window
    REPO = Path.cwd().parent
RAW_DIR = REPO / "data" / "raw" / "landsat"
OUT_GDB = REPO / "outputs" / "coastline.gdb"
TAG = f"{START:%Y%m%d}_{END:%Y%m%d}"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_GDB.parent.mkdir(parents=True, exist_ok=True)
if not arcpy.Exists(str(OUT_GDB)):
    arcpy.management.CreateFileGDB(str(OUT_GDB.parent), OUT_GDB.name)
arcpy.env.overwriteOutput = True


def gdb(name):
    """Path of a dataset inside the output geodatabase."""
    return str(OUT_GDB / name)

In [ ]:
gis = GIS(*GIS_PROFILE)
landsat = ImageryLayer(LANDSAT_URL)

In [8]:
def make_aoi_buffer_fs(lon, lat, radius_m, out_wkid=WEBMERCATOR, gis=None):
    """Circular AOI of radius_m metres around a lon/lat point."""
    pt = Point({"x": lon, "y": lat, "spatialReference": {"wkid": WGS84}})
    geom = buffer(
        [pt],
        in_sr=WGS84,
        distances=[radius_m],
        unit=M_ESRI,
        out_sr=out_wkid,
        geodesic=True,
        gis=gis,
    )
    return geom[0]

def make_aoi_fs(geom, out_wkid=WEBMERCATOR):
    """AOI FeatureSet from an Esri polygon geometry dict."""
    fs = FeatureSet.from_dict({
        "geometryType": "esriGeometryPolygon",
        "spatialReference": {"wkid": out_wkid},
        "fields": [{"name": "Name", "type": "esriFieldTypeString", "alias": "Name"}],
        "features": [{"attributes": {"Name": "AOI"}, "geometry": geom}]
    })
    return fs

def extent_from_geometry(geom, pad=0.05, wkid=WEBMERCATOR_LATEST, latest_wkid=WEBMERCATOR):
    """Padded extent dict (same format as the tutorial) from an Esri polygon with 'rings'."""
    xs = [pt[0] for ring in geom["rings"] for pt in ring]
    ys = [pt[1] for ring in geom["rings"] for pt in ring]
    dx = (max(xs) - min(xs)) * pad
    dy = (max(ys) - min(ys)) * pad
    return {
        "spatialReference": {"latestWkid": latest_wkid, "wkid": wkid},
        "xmin": min(xs) - dx,
        "ymin": min(ys) - dy,
        "xmax": max(xs) + dx,
        "ymax": max(ys) + dy,
    }

aoi_geom = make_aoi_buffer_fs(*BARWON_HEADS, radius_m=BUFFER_M, gis=gis)
aoi = make_aoi_fs(aoi_geom)

m = gis.map('Barwon Heads, Victoria, Australia')
m.content.add(aoi)

extent = extent_from_geometry(aoi_geom)
m.extent = extent

In [9]:
selected = landsat.filter_by(
    where=f"(Category = 1) and (CloudCover<{MAX_CLOUD})",
    time=[START, END],
    geometry=arcgis.geometry.filters.intersects(aoi_geom),
)
df = selected.query(
    out_fields="AcquisitionDate, GroupName, CloudCover, DayOfYear, SensorName",
    order_by_fields="CloudCover",
).sdf

df["AcquisitionDate"] = pd.to_datetime(df["AcquisitionDate"], unit="ms")
oid = df["OBJECTID"].tolist()
df

,OBJECTID,AcquisitionDate,GroupName,CloudCover,DayOfYear,SensorName,SHAPE
0,2222160,2017-03-18 00:09:44,LC08_L1TP_093087_20170318_20200904_02_T1_MTL,0.0002,87,Landsat 8,"{""rings"": [[[16167833.772, -4616470.0473], [16..."
1,2222155,2017-11-29 00:09:47,LC08_L1TP_093086_20171129_20200902_02_T1_MTL,0.0005,86,Landsat 8,"{""rings"": [[[16211896.338799998, -4415154.6305..."
2,2217923,2017-01-22 00:03:59,LC08_L1TP_092087_20170122_20200905_02_T1_MTL,0.0006,87,Landsat 8,"{""rings"": [[[16337530.451200001, -4616282.3621..."
3,2222154,2017-11-13 00:09:54,LC08_L1TP_093086_20171113_20200902_02_T1_MTL,0.0006,86,Landsat 8,"{""rings"": [[[16188109.884499997, -4506078.5562..."
4,2925228,2019-08-31 00:09:50,LC08_L1TP_093086_20190831_20200826_02_T1_MTL,0.0016,86,Landsat 8,"{""rings"": [[[16178250.228100002, -4557940.9162..."
5,2221230,2016-04-25 00:03:22,LC08_L1TP_092087_20160425_20200907_02_T1_MTL,0.002,87,Landsat 8,"{""rings"": [[[16338147.7042, -4616284.339600000..."
6,2222139,2017-03-18 00:09:20,LC08_L1TP_093086_20170318_20200904_02_T1_MTL,0.0024,86,Landsat 8,"{""rings"": [[[16212960.103200004, -4415049.5586..."
7,2222136,2017-01-29 00:09:42,LC08_L1TP_093086_20170129_20200905_02_T1_MTL,0.0036,86,Landsat 8,"{""rings"": [[[16191115.251500003, -4505254.7124..."
8,2225659,2016-12-12 00:09:56,LC08_L1TP_093086_20161212_20200905_02_T1_MTL,0.0045,86,Landsat 8,"{""rings"": [[[16209674.9142, -4415146.853100002..."
9,1747596,2018-01-16 00:09:41,LC08_L1TP_093086_20180116_20200902_02_T1_MTL,0.0049,86,Landsat 8,"{""rings"": [[[16211003.655500002, -4415141.7776..."


In [ ]:
landsat.mosaic_by(method='lock_rasters', lock_rasters=oid)
landsat.extent = extent
landsat

In [13]:
sr_out = arcpy.SpatialReference(OUT_SR)
aoi_poly = arcpy.AsShape(dict(aoi_geom), True).projectAs(sr_out)

aoi_fc = gdb(f"aoi_{TAG}")
arcpy.management.CopyFeatures([aoi_poly], aoi_fc)

ext = aoi_poly.extent
xmin = float(np.floor(ext.XMin / CELL) * CELL)
ymin = float(np.floor(ext.YMin / CELL) * CELL)
xmax = float(np.ceil(ext.XMax / CELL) * CELL)
ymax = float(np.ceil(ext.YMax / CELL) * CELL)
ncols = int(round((xmax - xmin) / CELL))
nrows = int(round((ymax - ymin) / CELL))
print(f"Export grid: {ncols} x {nrows} cells")

tif_name = f"landsat_{TAG}.tif"
tif_path = RAW_DIR / tif_name

mosaic_rule = {
    "mosaicMethod": "esriMosaicLockRaster",
    "lockRasterIds": oid,
    "ascending": True,
    "mosaicOperation": "MT_FIRST",
}

landsat.export_image(
    bbox=[xmin, ymin, xmax, ymax],
    bbox_sr=OUT_SR,
    image_sr=OUT_SR,
    size=[ncols, nrows],
    export_format="tiff",
    interpolation="RSP_NearestNeighbor",
    band_ids=[BAND_IDS["blue"], BAND_IDS["red"], BAND_IDS["nir"]],
    mosaic_rule=mosaic_rule,
    rendering_rule={"rasterFunction": "None"},
    f="image",
    save_folder=str(RAW_DIR),
    save_file=tif_name,
    adjust_aspect_ratio=False,
)
assert tif_path.exists(), f"Export did not produce {tif_path}"

arr = arcpy.RasterToNumPyArray(arcpy.Raster(str(tif_path)), nodata_to_value=-9999)
if arr.ndim != 3:
    raise RuntimeError(f"Expected a multiband export, got array shape {arr.shape}")
if arr.shape[0] == 3:
    blue, red, nir = arr
elif arr.shape[0] == 11:
    blue, red, nir = (arr[BAND_IDS[k]] for k in ("blue", "red", "nir"))
else:
    raise RuntimeError(f"Unexpected band count {arr.shape[0]}")
if blue.shape != (nrows, ncols):
    raise RuntimeError(f"Export size {blue.shape} != requested {(nrows, ncols)}")

Export grid: 667 x 667 cells


In [14]:
valid = blue > 0
water = ((blue > nir) & (blue > red) & valid).astype(np.uint8)
water[~valid] = 255

print("band means (valid cells):",
      {k: round(float(v[valid].mean()), 1) for k, v in (("blue", blue), ("red", red), ("nir", nir))})
water_frac = float((water == 1).sum() / max(valid.sum(), 1))
print(f"water fraction: {water_frac:.1%}")
if water_frac < 0.02 or water_frac > 0.98:
    print("WARNING: almost no water or almost all water. Check BAND_IDS and the scene.")

# [NEW] Write the binary array to a raster with the right location and projection
water_raw = gdb(f"water_binary_raw_{TAG}")
ras = arcpy.NumPyArrayToRaster(
    water,
    arcpy.Point(xmin, ymin),
    CELL,
    CELL,
    value_to_nodata=255,
)
ras.save(water_raw)
arcpy.management.DefineProjection(water_raw, sr_out)

band means (valid cells): {'blue': 1088.6, 'red': 710.1, 'nir': 1365.2}
water fraction: 45.2%


<Result 'c:\\Users\\sabastianbouma\\Documents\\coastline_mapping\\outputs\\coastline.gdb\\water_binary_raw_20160101_20191231'>

In [15]:
water_clip = gdb(f"water_binary_{TAG}")
arcpy.management.Clip(
    in_raster=water_raw,
    out_raster=water_clip,
    in_template_dataset=aoi_fc,
    nodata_value="255",
    clipping_geometry="ClippingGeometry",
    maintain_clipping_extent="NO_MAINTAIN_EXTENT",
)

poly_all = gdb(f"polygons_all_{TAG}")
arcpy.conversion.RasterToPolygon(
    in_raster=water_clip,
    out_polygon_features=poly_all,
    simplify="SIMPLIFY",
    raster_field="Value",
    create_multipart_features="SINGLE_OUTER_PART",
)

water_poly = gdb(f"water_polygons_{TAG}")
arcpy.analysis.Select(poly_all, water_poly, "gridcode = 1")

<Result 'c:\\Users\\sabastianbouma\\Documents\\coastline_mapping\\outputs\\coastline.gdb\\water_polygons_20160101_20191231'>

In [16]:
oid_field = arcpy.Describe(water_poly).OIDFieldName
with arcpy.da.SearchCursor(water_poly, ["OID@", "SHAPE@AREA"]) as cur:
    largest_oid, largest_area = max(cur, key=lambda r: r[1])
print(f"Largest water polygon: {largest_area / 1e6:.2f} km2")

coast_poly = gdb(f"coast_polygon_{TAG}")
arcpy.analysis.Select(water_poly, coast_poly, f"{oid_field} = {largest_oid}")

Largest water polygon: 135.68 km2


<Result 'c:\\Users\\sabastianbouma\\Documents\\coastline_mapping\\outputs\\coastline.gdb\\coast_polygon_20160101_20191231'>

In [17]:
coast_line_raw = gdb(f"coastline_polyline_{TAG}")
PolygonToLine(coast_poly, coast_line_raw, "IGNORE_NEIGHBORS")

<Result 'c:\\Users\\sabastianbouma\\Documents\\coastline_mapping\\outputs\\coastline.gdb\\coastline_polyline_20160101_20191231'>

In [18]:
mask_poly = gdb(f"mask_{TAG}")
arcpy.management.Dissolve(poly_all, mask_poly)

mask_neg = gdb(f"mask_neg_buffer_{TAG}")
arcpy.analysis.Buffer(mask_poly, mask_neg, "-10 Meters")

coastline_final = gdb(f"extracted_coastline_{TAG}")
arcpy.analysis.Clip(coast_line_raw, mask_neg, coastline_final)
print("Coastline written to:", coastline_final)

Coastline written to: c:\Users\sabastianbouma\Documents\coastline_mapping\outputs\coastline.gdb\extracted_coastline_20160101_20191231


In [19]:
m9 = gis.map("Barwon Heads, Victoria, Australia")
m9

In [20]:
m9.center = [BARWON_HEADS[1], BARWON_HEADS[0]]
m9.zoom = 13
m9.content.add(landsat)
coast_3857 = gdb(f"extracted_coastline_{TAG}_3857")
arcpy.management.Project(coastline_final, coast_3857, arcpy.SpatialReference(WEBMERCATOR))
dfm_1 = pd.DataFrame.spatial.from_featureclass(coast_3857)

dfm_1.spatial.plot(map_widget=m9,
                   renderer_type="s",
                   pallete="spring",
                   alpha=1)

True